下面是本地已经有的的历史数据解析和展示

In [ ]:
#快速回测所需代码

import sys,pandas,numpy,os
from pathlib import Path
#以下是windows环境下的路径
project_path = Path("d:/360MoveData/Users/Lenovo/Desktop/cuhksz/quant_backtest_claude/examples/self_test.ipynb").parent.parent
#将项目路径插入系统寻找路径首位
#project_path = Path.cwd().parent.parent
print(str(project_path))
sys.path.insert(0,str(project_path))
#导入数据处理模块的类
#查看当前目录
os.chdir(str(project_path))
print(os.getcwd())
from data.loader import TushareDataLoader,DataManager


d:\360MoveData\Users\Lenovo\Desktop\cuhksz\quant_backtest_claude
d:\360MoveData\Users\Lenovo\Desktop\cuhksz\quant_backtest_claude


In [6]:
dm = DataManager()
dm.load('20250101','20251219')  # 加载close/high_limit/vol5/is_st/paused等数据
df = dm.price
df

2026-02-25 19:52:18,541 - INFO - TushareDataLoader 初始化完成，数据目录: d:\360MoveData\Users\Lenovo\Desktop\cuhksz\quant_backtest_claude\storage
2026-02-25 19:52:20,759 - INFO - 加载日线数据: 13606806 条, 5739 只股票, 3916 个交易日
2026-02-25 19:52:24,022 - INFO - 加载每日指标: 13515700 条
2026-02-25 19:52:24,540 - INFO - DataManager 加载完成: 20250101 ~ 20251219
2026-02-25 19:52:24,597 - INFO -   日线: 1270277 条, 5490 只股票


ts_code,000001.SZ,000002.SZ,000004.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,000011.SZ,000012.SZ,...,920964.BJ,920970.BJ,920971.BJ,920974.BJ,920976.BJ,920978.BJ,920981.BJ,920982.BJ,920985.BJ,920992.BJ
trade_date,,,,,,,,,,,,,,,,,,,,,
2025-01-02,11.43,7.11,14.18,7.25,7.02,2.84,8.90,2.71,8.62,5.14,...,6.93,5.65,26.50,6.94,18.80,13.30,25.70,205.90,9.89,13.29
2025-01-03,11.38,7.00,12.80,6.91,6.60,2.65,8.72,2.64,8.27,5.08,...,6.80,5.77,26.16,7.27,18.51,13.26,26.00,209.32,9.82,12.95
2025-01-06,11.44,6.98,12.52,6.70,6.63,2.58,8.74,2.64,8.30,5.12,...,6.49,5.80,24.86,7.25,17.56,13.18,26.99,216.62,10.04,13.15
2025-01-07,11.51,7.05,13.11,6.85,6.93,2.66,8.73,2.90,8.53,5.09,...,6.60,5.91,25.40,7.69,18.11,13.54,27.57,214.51,10.18,13.27
2025-01-08,11.50,6.96,13.40,6.96,6.93,2.64,8.63,2.96,8.45,5.06,...,6.62,5.95,27.18,7.63,18.36,13.83,27.54,217.72,10.09,13.57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-15,11.51,4.87,10.82,9.82,11.61,2.91,10.04,3.53,9.03,4.45,...,6.82,8.09,32.40,8.11,25.52,35.06,32.44,262.99,7.43,17.95
2025-12-16,11.48,4.97,10.43,9.36,12.00,2.86,9.80,3.42,8.90,4.43,...,6.86,8.02,32.52,8.12,26.04,34.80,33.08,262.15,7.48,18.09
2025-12-17,11.53,4.96,10.20,9.53,11.52,2.86,9.94,3.39,8.91,4.43,...,6.86,8.39,31.88,8.21,25.80,34.95,33.12,254.30,7.47,18.32


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('TkAgg')  # 或 'Qt5Agg'，根据你的系统选择
from datetime import datetime

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

def load_and_preprocess_data(file_path):
    """
    加载并预处理CSV数据
    """
    # 尝试不同的编码格式
    encodings = ['gbk', 'gb2312', 'gb18030', 'utf-8-sig', 'latin1']
    
    for encoding in encodings:
        try:
            print(f"尝试使用 {encoding} 编码读取...")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"成功使用 {encoding} 编码读取！")
            break
        except UnicodeDecodeError:
            continue
    else:
        # 如果所有编码都失败，使用最宽松的 latin1 编码
        print("所有尝试都失败，使用 latin1 编码...")
        df = pd.read_csv(file_path, encoding='latin1')
    
    # 去除股票代码中的引号
    df['股票代码'] = df['股票代码'].astype(str).str.strip("'")
    
    # 转换日期格式
    df['起始时间'] = pd.to_datetime(df['起始时间'], format='%Y%m%d')
    df['终止时间'] = pd.to_datetime(df['终止时间'], format='%Y%m%d')
    
    # 计算持有天数
    df['持有天数'] = (df['终止时间'] - df['起始时间']).dt.days
    
    # 确保涨跌幅是数值类型
    df['区间涨跌幅'] = df['区间涨跌幅'].astype(str).str.replace('%', '').astype(float) / 100
    
    return df

def analyze_stock_performance(df):
    """
    分析各股票的表现
    """
    # 按股票分组分析
    stock_performance = df.groupby(['股票代码', '股票简称']).agg({
        '区间涨跌幅': ['mean', 'std', 'count', 'sum'],
        '起始价': 'first',
        '终止价': 'last'
    }).round(4)
    
    stock_performance.columns = ['平均涨跌幅', '涨跌幅标准差', '出现次数', '累计涨跌幅', '起始价', '终止价']
    stock_performance = stock_performance.reset_index()
    
    # 按平均涨跌幅排序
    stock_performance = stock_performance.sort_values('平均涨跌幅', ascending=False)
    
    return stock_performance

def calculate_daily_returns_with_position_constraint(df, initial_capital=30000):
    """
    计算每日策略收益，考虑最小交易单位约束
    """
    # 获取所有交易日
    all_dates = pd.date_range(
        start=df['起始时间'].min(), 
        end=df['终止时间'].max(), 
        freq='D'
    )
    
    # 初始化每日持仓和收益
    daily_positions = {}
    daily_returns = []
    daily_portfolio = []
    
    # 按日期排序
    df_sorted = df.sort_values('起始时间')
    
    # 为每一笔交易创建持仓记录
    for idx, row in df_sorted.iterrows():
        stock_code = row['股票代码']
        start_date = row['起始时间']
        end_date = row['终止时间']
        
        # 计算该持仓期间的每日收益
        date_range = pd.date_range(start=start_date, end=end_date, freq='D')
        
        # 根据区间涨跌幅估算每日收益率（简化：假设均匀分布）
        total_return = row['区间涨跌幅']
        days = len(date_range)
        if days > 1:
            daily_return_rate = (1 + total_return) ** (1/days) - 1
        else:
            daily_return_rate = total_return
        
        for i, date in enumerate(date_range):
            if date not in daily_positions:
                daily_positions[date] = []
            
            # 记录该股票当日的持仓信息
            daily_positions[date].append({
                'stock_code': stock_code,
                'stock_name': row['股票简称'],
                'entry_price': row['起始价'] if i == 0 else None,
                'daily_return': daily_return_rate
            })
    
    # 计算每日等权收益，考虑最小交易单位
    capital = initial_capital
    portfolio_values = []
    
    for date in all_dates:
        if date in daily_positions:
            positions = daily_positions[date]
            num_stocks = len(positions)
            
            # 计算等权分配每只股票的资金
            capital_per_stock = capital / num_stocks
            
            # 检查是否够买至少1手（100股）
            valid_positions = []
            for pos in positions:
                # 使用起始价作为参考（这里简化处理，实际应该用当日开盘价）
                # 这里假设用该股票在该期间的起始价
                stock_price = df_sorted[
                    (df_sorted['股票代码'] == pos['stock_code']) & 
                    (df_sorted['起始时间'] <= date) & 
                    (df_sorted['终止时间'] >= date)
                ]['起始价'].values
                
                if len(stock_price) > 0:
                    price = stock_price[0]
                    if capital_per_stock >= price * 100:  # 够买1手
                        valid_positions.append(pos)
            
            # 重新计算实际可投资的股票数量和资金分配
            if valid_positions:
                actual_num_stocks = len(valid_positions)
                actual_capital_per_stock = capital / actual_num_stocks
                
                # 计算当日组合收益
                daily_return = sum([pos['daily_return'] for pos in valid_positions]) / actual_num_stocks
                daily_returns.append(daily_return)
                
                # 更新资本
                capital = capital * (1 + daily_return)
                portfolio_values.append(capital)
                
                # 记录当日持仓信息
                daily_portfolio.append({
                    'date': date,
                    'num_stocks_total': num_stocks,
                    'num_stocks_tradable': actual_num_stocks,
                    'capital_per_stock': actual_capital_per_stock,
                    'daily_return': daily_return,
                    'portfolio_value': capital
                })
            else:
                # 没有可交易的股票
                daily_returns.append(0)
                portfolio_values.append(capital)
                daily_portfolio.append({
                    'date': date,
                    'num_stocks_total': num_stocks,
                    'num_stocks_tradable': 0,
                    'capital_per_stock': 0,
                    'daily_return': 0,
                    'portfolio_value': capital
                })
        else:
            # 没有持仓的日子
            daily_returns.append(0)
            portfolio_values.append(capital)
            daily_portfolio.append({
                'date': date,
                'num_stocks_total': 0,
                'num_stocks_tradable': 0,
                'capital_per_stock': 0,
                'daily_return': 0,
                'portfolio_value': capital
            })
    
    # 创建结果DataFrame
    portfolio_df = pd.DataFrame(daily_portfolio)
    
    return portfolio_df, daily_returns, portfolio_values

def plot_results(stock_performance, portfolio_df):
    """
    可视化结果
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 股票表现排名（Top 20）
    top20 = stock_performance.head(20)
    axes[0, 0].barh(range(len(top20)), top20['平均涨跌幅'].values * 100)
    axes[0, 0].set_yticks(range(len(top20)))
    axes[0, 0].set_yticklabels([f"{code}\n{name}" for code, name in zip(top20['股票代码'], top20['股票简称'])])
    axes[0, 0].set_yticklabels([f"{name}" for name in top20['股票简称']])
    axes[0, 0].set_xlabel('平均涨跌幅 (%)')
    axes[0, 0].set_title('表现最好的20只股票')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. 股票出现次数分布
    axes[0, 1].hist(stock_performance['出现次数'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('出现次数')
    axes[0, 1].set_ylabel('股票数量')
    axes[0, 1].set_title('股票出现次数分布')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. 净值曲线
    axes[1, 0].plot(portfolio_df['date'], portfolio_df['portfolio_value'], 'b-', linewidth=2)
    axes[1, 0].set_xlabel('日期')
    axes[1, 0].set_ylabel('资产净值 (元)')
    axes[1, 0].set_title('策略净值曲线')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # 4. 每日持仓数量
    axes[1, 1].plot(portfolio_df['date'], portfolio_df['num_stocks_tradable'], 'g-', linewidth=2, label='实际可交易')
    axes[1, 1].plot(portfolio_df['date'], portfolio_df['num_stocks_total'], 'r--', linewidth=1, alpha=0.7, label='理论持仓')
    axes[1, 1].set_xlabel('日期')
    axes[1, 1].set_ylabel('股票数量')
    axes[1, 1].set_title('每日持仓数量')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

def calculate_performance_metrics(portfolio_df, initial_capital=30000):
    """
    计算策略性能指标
    """
    final_value = portfolio_df['portfolio_value'].iloc[-1]
    total_return = (final_value - initial_capital) / initial_capital * 100
    
    # 计算年化收益率（假设252个交易日）
    days = len(portfolio_df)
    annual_return = ((final_value / initial_capital) ** (252 / days) - 1) * 100 if days > 0 else 0
    
    # 计算最大回撤
    cumulative = portfolio_df['portfolio_value'].values
    running_max = np.maximum.accumulate(cumulative)
    drawdown = (cumulative - running_max) / running_max * 100
    max_drawdown = drawdown.min()
    
    # 计算夏普比率（假设无风险利率为3%）
    daily_returns = portfolio_df['daily_return'].values
    risk_free_rate = 0.03 / 252
    excess_returns = daily_returns - risk_free_rate
    sharpe_ratio = np.sqrt(252) * excess_returns.mean() / excess_returns.std() if excess_returns.std() > 0 else 0
    
    metrics = {
        '初始资金': f'{initial_capital:,.0f} 元',
        '最终资金': f'{final_value:,.2f} 元',
        '总收益率': f'{total_return:.2f}%',
        '年化收益率': f'{annual_return:.2f}%',
        '最大回撤': f'{max_drawdown:.2f}%',
        '夏普比率': f'{sharpe_ratio:.3f}',
        '交易天数': len(portfolio_df),
        '平均每日持仓': f'{portfolio_df["num_stocks_tradable"].mean():.1f} 只'
    }
    
    return metrics

def main():
    # 文件路径（请修改为你的实际文件路径）
    file_path = r"D:\360MoveData\Users\Lenovo\Desktop\连板龙头26年回测.csv"  # 修改这里
    
    # 加载数据
    print("正在加载数据...")
    df = load_and_preprocess_data(file_path)
    
    print(f"数据加载完成，共 {len(df)} 条记录")
    print(f"时间范围：{df['起始时间'].min()} 至 {df['终止时间'].max()}")
    print(f"涉及股票数量：{df['股票代码'].nunique()} 只")
    
    # 分析股票表现
    print("\n正在分析股票表现...")
    stock_performance = analyze_stock_performance(df)
    
    # 显示表现最好的10只股票
    print("\n表现最好的10只股票：")
    print(stock_performance.head(10).to_string(index=False))
    
    # 显示表现最差的10只股票
    print("\n表现最差的10只股票：")
    print(stock_performance.tail(10).to_string(index=False))
    
    # 计算每日收益和净值
    print("\n正在计算策略收益（考虑最小交易单位）...")
    portfolio_df, daily_returns, portfolio_values = calculate_daily_returns_with_position_constraint(df)
    
    # 计算性能指标
    metrics = calculate_performance_metrics(portfolio_df)
    
    print("\n" + "="*50)
    print("策略性能指标：")
    print("="*50)
    for key, value in metrics.items():
        print(f"{key}: {value}")
    
    # 可视化结果
    print("\n正在生成可视化图表...")
    plot_results(stock_performance, portfolio_df)
    
    # 保存结果到CSV
    output_file = 'strategy_analysis_results.csv'
    portfolio_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n详细结果已保存到: {output_file}")
    
    # 保存股票表现分析
    stock_file = 'stock_performance_analysis.csv'
    stock_performance.to_csv(stock_file, index=False, encoding='utf-8-sig')
    print(f"股票表现分析已保存到: {stock_file}")

if __name__ == "__main__":
    main()

正在加载数据...
尝试使用 gbk 编码读取...
成功使用 gbk 编码读取！
数据加载完成，共 293 条记录
时间范围：2026-01-05 00:00:00 至 2026-02-25 00:00:00
涉及股票数量：166 只

正在分析股票表现...

表现最好的10只股票：
  股票代码 股票简称  平均涨跌幅  涨跌幅标准差  出现次数  累计涨跌幅   起始价   终止价
603078  江化微 0.2101     NaN     1 0.2101 23.56 28.51
001896 豫能控股 0.2096  0.0004     2 0.4193  9.12 10.03
002716 湖南白银 0.2053     NaN     1 0.2053 15.00 18.08
600397 江钨装备 0.1908     NaN     1 0.1908 14.31 17.04
002498 汉缆股份 0.1808  0.0329     3 0.5424  6.77  5.69
601611 中国核建 0.1778     NaN     1 0.1778 16.93 19.94
603926 铁流股份 0.1706     NaN     1 0.1706 22.21 26.00
600676 交运股份 0.1676  0.0582     2 0.3353  8.22  9.03
603618 杭电股份 0.1633  0.0398     3 0.4899 12.02 12.02
002413 雷科防务 0.1566  0.0578     3 0.4698 16.71 16.71

表现最差的10只股票：
  股票代码 股票简称   平均涨跌幅  涨跌幅标准差  出现次数   累计涨跌幅   起始价   终止价
002030 达安基因 -0.1089     NaN     1 -0.1089  7.71  6.87
603095 越剑智能 -0.1106     NaN     1 -0.1106 21.60 19.21
000737 北方铜业 -0.1473     NaN     1 -0.1473 20.57 17.54
002219  新里程 -0.1491     NaN     1 -0.1491  2.75  2.34


In [1]:
import rqdatac
rqdatac.init()

c:\Users\Lenovo\anaconda3\envs\backtest-env\lib\site-packages\rqdatac\client.py:263: UserWarning: Your account will be expired after  27 days. Please call us at 0755-22676337 to upgrade or purchase or renew your contract.
  warnings.warn("Your account will be expired after  {} days. "


In [ ]:
rqsdk download-data --sample